# 01 — Data Prep & Unified Features

End-to-end, fully visible pipeline that produces:

- `panel_weekly.parquet`  — Monday-aligned weekly panel for every observed (nest, facility, product) pair. Columns: pair keys, `week_start`, `weekly_units`, `demand_class`, `tier`, `split`, `mean_nonzero`, `nonzero_weeks`. Used by the statistical baseline (Notebook 02) and by Notebook 04 for the rare-pair heuristic and the next-week shopping list.
- `unified_features.parquet` — supervised feature table for `tier == 'modellable'` pairs only. Used by Notebook 03 (LightGBM + XGBoost). Both ML models share the same feature columns.

We use **parquet** (not CSV) for intermediate files because they are ~10x smaller and much faster to read/write — important since the panel has ~2.5M rows. The final shopping list (Notebook 04) still goes out as CSV for easy inspection.

**Conventions used everywhere downstream:**

- Grain: `NEST_NAME × HEALTH_FACILITY_NAME × PRODUCT_NAME × week_start` (Monday-aligned).
- Per-nest calendar split: each nest's last 5 weeks → `test`, prior 5 weeks → `valid`, rest → `train`.
- Target convention: predict `weekly_units` of week W using only information available at the end of week W-1 (every lag/rolling/history feature is shifted accordingly).
- 4 nests: GH1 Omenako, GH3 Vobsi, RW1 Muhanga, RW2 Kayonza.


## Step 0 — Imports and paths

In [2]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)

import os
# Default data dir
DATA_DIR = Path.cwd() / 'data'

# Search for the files in Google Drive
drive_path = Path('/content/drive/MyDrive')
target_file = 'GH-1 2024 2025 2026 Demand.csv'

print("Searching for data files in Google Drive, this might take a moment...")
if drive_path.exists():
    for root, dirs, files in os.walk(drive_path):
        if target_file in files:
            DATA_DIR = Path(root)
            break

print(f"\nUsing DATA_DIR: {DATA_DIR}\n")

RAW_FILES = [
    DATA_DIR / 'GH-1 2024 2025 2026 Demand.csv',
    DATA_DIR / 'GH-3 2024 2025 2026 Demand.csv',
    DATA_DIR / 'Copy of RW-1 2025 2026.csv',
    DATA_DIR / 'Copy of RW-2 2025 2026.csv',
]
PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']
CORE_COLS = ['DATE', 'NEST_NAME', 'SHIPMENT_PRIORITY', 'HEALTH_FACILITY_NAME',
             'UNITS_DELIVERED', 'PRODUCT_NAME']

# Planned shipment priorities only (drop reactive 'emergency').
PRIORITY_SCOPE = ['custom window', 'resupply', 'scheduled', 'fast']

# ADI-CV2 thresholds (Syntetos-Boylan).
ADI_CUTOFF = 1.32
CV2_CUTOFF = 0.49

# Tier rule for 'modellable' pairs.
MIN_NONZERO_WEEKS  = 5
MAX_ADI            = 15.0
RECENCY_WEEKS      = 26

# Per-nest calendar split sizes.
N_TEST_WEEKS  = 5
N_VALID_WEEKS = 5

t0 = time.time()
print('Inputs:')
for p in RAW_FILES:
    print(f'  {p.name:<40s}  EXISTS={p.exists()}')


Searching for data files in Google Drive, this might take a moment...

Using DATA_DIR: /content/drive/MyDrive/Spring2026/Analytics in Practice - Zipline Demand Forecasting

Inputs:
  GH-1 2024 2025 2026 Demand.csv            EXISTS=True
  GH-3 2024 2025 2026 Demand.csv            EXISTS=True
  Copy of RW-1 2025 2026.csv                EXISTS=True
  Copy of RW-2 2025 2026.csv                EXISTS=True


## Step 1 — Load and concatenate the 4 raw CSVs

In [6]:
frames = []
for p in RAW_FILES:
    d = pd.read_csv(p, usecols=CORE_COLS, parse_dates=['DATE'], low_memory=False)
    print(f'  loaded {p.name:<40s} rows={len(d):,}')
    frames.append(d)
df_raw = pd.concat(frames, ignore_index=True)
print(f'\nConcatenated raw shape: {df_raw.shape}   elapsed={time.time()-t0:.1f}s')


  loaded GH-1 2024 2025 2026 Demand.csv           rows=86,300
  loaded GH-3 2024 2025 2026 Demand.csv           rows=98,953
  loaded Copy of RW-1 2025 2026.csv               rows=214,573
  loaded Copy of RW-2 2025 2026.csv               rows=145,531

Concatenated raw shape: (545357, 6)   elapsed=31.0s


## Step 2 — Clean (drop nulls, normalize strings, drop emergency priorities)

In [7]:
df = df_raw.dropna(subset=['HEALTH_FACILITY_NAME', 'PRODUCT_NAME', 'UNITS_DELIVERED', 'NEST_NAME']).copy()
for c in PAIR_COLS:
    df[c] = df[c].astype(str).str.strip()
before = len(df)
df = df[df['SHIPMENT_PRIORITY'].isin(PRIORITY_SCOPE)].copy()
df['units'] = df['UNITS_DELIVERED'].astype(float)
print(f'After cleaning + priority filter: {before:,} -> {len(df):,} rows')
print('Per-nest delivery counts:')
print(df['NEST_NAME'].value_counts())
print(f'elapsed={time.time()-t0:.1f}s')


After cleaning + priority filter: 544,168 -> 443,514 rows
Per-nest delivery counts:
NEST_NAME
RW1 Muhanga    152998
RW2 Kayonza    113153
GH3 Vobsi       94734
GH1 Omenako     82629
Name: count, dtype: int64
elapsed=33.8s


## Step 3 — Build the Monday-aligned weekly panel (zero-filled per nest)

Zero-fill is done **per nest** so a GH pair is not given fake zeros in the period before GH started.

In [8]:
dt = pd.to_datetime(df['DATE'])
df['week_start'] = (dt - pd.to_timedelta(dt.dt.dayofweek, unit='D')).dt.normalize()

weekly = (df.groupby(PAIR_COLS + ['week_start'], as_index=False)['units']
            .sum().rename(columns={'units': 'weekly_units'}))

panels = []
for nest, g in weekly.groupby('NEST_NAME', sort=False):
    weeks = pd.date_range(g['week_start'].min(), g['week_start'].max(), freq='W-MON')
    pairs = g[PAIR_COLS].drop_duplicates()
    full = pairs.merge(pd.DataFrame({'week_start': weeks}), how='cross')
    full = full.merge(g, on=PAIR_COLS + ['week_start'], how='left')
    full['weekly_units'] = full['weekly_units'].fillna(0.0)
    panels.append(full)
panel = pd.concat(panels, ignore_index=True).sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)

print(f'Weekly panel shape: {panel.shape}')
print('Pairs per nest:')
print(panel.groupby('NEST_NAME')[PAIR_COLS].apply(lambda d: d.drop_duplicates().shape[0]))
print('Week range per nest:')
print(panel.groupby('NEST_NAME')['week_start'].agg(['min', 'max', 'nunique']))
print(f'elapsed={time.time()-t0:.1f}s')


Weekly panel shape: (4091010, 5)
Pairs per nest:
NEST_NAME
GH1 Omenako     9296
GH3 Vobsi       8667
RW1 Muhanga    18930
RW2 Kayonza    11463
dtype: int64
Week range per nest:
                   min        max  nunique
NEST_NAME                                 
GH1 Omenako 2024-01-01 2026-02-09      111
GH3 Vobsi   2024-01-01 2026-02-09      111
RW1 Muhanga 2024-12-30 2026-04-20       69
RW2 Kayonza 2024-12-30 2026-04-20       69
elapsed=51.0s


## Step 4 — Per-nest calendar split (last 5 weeks = test, prior 5 weeks = valid)

In [9]:
panel['nest_max_week'] = panel.groupby('NEST_NAME')['week_start'].transform('max')
panel['test_lo']  = panel['nest_max_week'] - pd.Timedelta(weeks=N_TEST_WEEKS - 1)
panel['valid_lo'] = panel['test_lo']        - pd.Timedelta(weeks=N_VALID_WEEKS)

panel['split'] = np.select(
    [panel['week_start'] >= panel['test_lo'],
     panel['week_start'] >= panel['valid_lo']],
    ['test', 'valid'],
    default='train',
)
panel = panel.drop(columns=['nest_max_week', 'test_lo', 'valid_lo'])

print('Split row counts:')
print(panel['split'].value_counts())
print('\nSplit weeks per nest:')
print(panel.groupby(['NEST_NAME', 'split'])['week_start'].agg(['min', 'max', 'nunique']))


Split row counts:
split
train    3607450
valid     241780
test      241780
Name: count, dtype: int64

Split weeks per nest:
                         min        max  nunique
NEST_NAME   split                               
GH1 Omenako test  2026-01-12 2026-02-09        5
            train 2024-01-01 2025-12-01      101
            valid 2025-12-08 2026-01-05        5
GH3 Vobsi   test  2026-01-12 2026-02-09        5
            train 2024-01-01 2025-12-01      101
            valid 2025-12-08 2026-01-05        5
RW1 Muhanga test  2026-03-23 2026-04-20        5
            train 2024-12-30 2026-02-09       59
            valid 2026-02-16 2026-03-16        5
RW2 Kayonza test  2026-03-23 2026-04-20        5
            train 2024-12-30 2026-02-09       59
            valid 2026-02-16 2026-03-16        5


## Step 5 — ADI–CV² classification + tier label

For every pair we compute, using **training history only** (so labels do not leak from valid/test):

- `nonzero_weeks`, `adi = total_weeks / nonzero_weeks`, `cv2 = (std_nz / mean_nz)^2`, `mean_nonzero`.

Then we label:
- `demand_class` ∈ {Smooth, Erratic, Intermittent, Lumpy, Zero_only} (Syntetos–Boylan).
- `tier = 'modellable'` if `nonzero_weeks ≥ 5` AND `adi ≤ 15` AND last activity within `RECENCY_WEEKS`; else `'rare'`.

In [10]:
tr_panel = panel[panel['split'] == 'train']
pair_stats = (tr_panel.groupby(PAIR_COLS, as_index=False)
              .agg(total_weeks=('weekly_units', 'count'),
                   nonzero_weeks=('weekly_units', lambda x: int((x > 0).sum())),
                   mean_nonzero=('weekly_units', lambda x: float(x[x > 0].mean()) if (x > 0).any() else 0.0),
                   std_nonzero=('weekly_units',  lambda x: float(x[x > 0].std())  if (x > 0).sum() > 1 else 0.0)))
pair_stats['adi'] = pair_stats['total_weeks'] / pair_stats['nonzero_weeks'].replace(0, np.nan)
pair_stats['cv2'] = ((pair_stats['std_nonzero'] / pair_stats['mean_nonzero'].replace(0, np.nan)) ** 2).fillna(0)

# Demand class
def classify(r):
    if r['nonzero_weeks'] == 0: return 'Zero_only'
    if r['adi'] < ADI_CUTOFF:
        return 'Smooth' if r['cv2'] < CV2_CUTOFF else 'Erratic'
    return 'Intermittent' if r['cv2'] < CV2_CUTOFF else 'Lumpy'
pair_stats['demand_class'] = pair_stats.apply(classify, axis=1)

# Recency: per-nest max_week minus last_active_week
last_active = (panel.loc[panel['weekly_units'] > 0]
                    .groupby(PAIR_COLS, as_index=False)['week_start']
                    .max().rename(columns={'week_start': 'last_active_week'}))
nest_max = panel.groupby('NEST_NAME', as_index=False)['week_start'].max().rename(columns={'week_start': 'nest_max_week'})
pair_stats = pair_stats.merge(last_active, on=PAIR_COLS, how='left').merge(nest_max, on='NEST_NAME', how='left')
pair_stats['weeks_since_active'] = (pair_stats['nest_max_week'] - pair_stats['last_active_week']).dt.days / 7

pair_stats['tier'] = np.where(
    (pair_stats['nonzero_weeks'] >= MIN_NONZERO_WEEKS)
    & (pair_stats['adi'] <= MAX_ADI)
    & (pair_stats['weeks_since_active'] <= RECENCY_WEEKS),
    'modellable', 'rare',
)

panel = panel.merge(
    pair_stats[PAIR_COLS + ['demand_class', 'tier', 'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks']],
    on=PAIR_COLS, how='left',
)

print('Tier distribution (per pair):')
print(pair_stats['tier'].value_counts())
print('\nModellable pairs per nest:')
print(pair_stats.loc[pair_stats['tier'] == 'modellable'].groupby('NEST_NAME').size())
print(f'elapsed={time.time()-t0:.1f}s')


Tier distribution (per pair):
tier
rare          40066
modellable     8290
Name: count, dtype: int64

Modellable pairs per nest:
NEST_NAME
GH1 Omenako     848
GH3 Vobsi      1906
RW1 Muhanga    3554
RW2 Kayonza    1982
dtype: int64
elapsed=95.1s


## Step 6 — Save the lean weekly panel (parquet)

In [11]:
OUT_DIR = Path('/content')
panel_out = panel[PAIR_COLS + ['week_start', 'weekly_units', 'demand_class', 'tier', 'split',
                                'mean_nonzero', 'nonzero_weeks']]
panel_out.to_parquet(OUT_DIR / 'panel_weekly.parquet', index=False)
print(f'Saved panel_weekly.parquet  shape: {panel_out.shape}'
      f'  size={(OUT_DIR / "panel_weekly.parquet").stat().st_size/1e6:.1f} MB'
      f'  elapsed={time.time()-t0:.1f}s')

Saved panel_weekly.parquet  shape: (4091010, 10)  size=1.1 MB  elapsed=210.3s


## Step 7 — Build the unified feature table (modellable pairs only)

From here on we work only on `tier == 'modellable'` pairs. Every feature is **past-only**.

In [13]:
feat = panel[panel['tier'] == 'modellable'].copy()
feat = feat.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
print('Modellable feature frame starting shape:', feat.shape)


Modellable feature frame starting shape: (687678, 12)


### 7a. Calendar features

In [14]:
feat['week_of_year'] = feat['week_start'].dt.isocalendar().week.astype(int)
feat['month']        = feat['week_start'].dt.month
feat['quarter']      = feat['week_start'].dt.quarter
feat['week_sin']     = np.sin(2 * np.pi * feat['week_of_year'] / 52)
feat['week_cos']     = np.cos(2 * np.pi * feat['week_of_year'] / 52)
feat['rainy_season'] = feat['month'].between(4, 10).astype(int)


### 7b. Lag features  (1, 2, 3, 4, 8, 12 weeks)

In [15]:
g_units = feat.groupby(PAIR_COLS, sort=False)['weekly_units']
for k in [1, 2, 3, 4, 8, 12]:
    feat[f'lag_{k}'] = g_units.shift(k)
print(f'Lags done. elapsed={time.time()-t0:.1f}s')


Lags done. elapsed=329.0s


### 7c. Rolling features (windows = 4, 8, 13; mean / std / max / nonzero_rate)

All rolling features are computed on the **shifted-by-1** series so the current week is excluded from its own feature.

In [16]:
shifted_units = feat.groupby(PAIR_COLS, sort=False)['weekly_units'].shift(1)
shifted_occ   = (shifted_units > 0).astype(float)
keys = [feat[c] for c in PAIR_COLS]
for w in [4, 8, 13]:
    feat[f'roll_mean_{w}']    = shifted_units.groupby(keys).transform(lambda s: s.rolling(w, min_periods=1).mean())
    feat[f'roll_std_{w}']     = shifted_units.groupby(keys).transform(lambda s: s.rolling(w, min_periods=1).std().fillna(0))
    feat[f'roll_max_{w}']     = shifted_units.groupby(keys).transform(lambda s: s.rolling(w, min_periods=1).max())
    feat[f'roll_nz_rate_{w}'] = shifted_occ.groupby(keys).transform(lambda s: s.rolling(w, min_periods=1).mean())
print(f'Rolling done. elapsed={time.time()-t0:.1f}s')


Rolling done. elapsed=351.3s


### 7d. Intermittent-demand features (per pair, past-only inline loop)

In [17]:
ws_all, lnz_all, hnz_all, ca_all = [], [], [], []
for _, g in feat.groupby(PAIR_COLS, sort=False):
    units = g['weekly_units'].to_numpy()
    n = len(units)
    ws  = np.full(n, np.nan)
    lnz = np.full(n, np.nan)
    hnz = np.full(n, np.nan)
    ca  = np.zeros(n)
    last_idx, last_val, cum = -1, np.nan, 0
    for i in range(n):
        if i > 0:
            hnz[i] = cum / i
            ca[i]  = cum
            if last_idx >= 0:
                ws[i]  = i - last_idx
                lnz[i] = last_val
        if units[i] > 0:
            last_idx, last_val, cum = i, units[i], cum + 1
    ws_all.extend(ws); lnz_all.extend(lnz); hnz_all.extend(hnz); ca_all.extend(ca)

feat['weeks_since_last_demand'] = ws_all
feat['last_nonzero_units']      = lnz_all
feat['historical_nonzero_rate'] = hnz_all
feat['cumulative_active_weeks'] = ca_all
print(f'Intermittent done. elapsed={time.time()-t0:.1f}s')


Intermittent done. elapsed=368.8s


### 7e. Expanding history at facility / product / pair level (past-only)

Aggregate `weekly_units` to the desired level, then take `.shift(1).expanding().mean()` so the current week never sees itself.

In [18]:
# Facility-level history
fac = (feat.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], as_index=False)
            ['weekly_units'].sum().rename(columns={'weekly_units': 'fac_total'})
            .sort_values(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start']))
fac['fac_nz'] = (fac['fac_total'] > 0).astype(int)
gf = fac.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME'])
fac['facility_hist_mean']    = gf['fac_total'].transform(lambda s: s.shift(1).expanding().mean())
fac['facility_hist_nz_rate'] = gf['fac_nz'   ].transform(lambda s: s.shift(1).expanding().mean())
feat = feat.merge(fac[['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start',
                       'facility_hist_mean', 'facility_hist_nz_rate']],
                  on=['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], how='left')

# Product-level history (within nest)
prod = (feat.groupby(['NEST_NAME', 'PRODUCT_NAME', 'week_start'], as_index=False)
             ['weekly_units'].sum().rename(columns={'weekly_units': 'prod_total'})
             .sort_values(['NEST_NAME', 'PRODUCT_NAME', 'week_start']))
prod['prod_nz'] = (prod['prod_total'] > 0).astype(int)
gp = prod.groupby(['NEST_NAME', 'PRODUCT_NAME'])
prod['product_hist_mean']    = gp['prod_total'].transform(lambda s: s.shift(1).expanding().mean())
prod['product_hist_nz_rate'] = gp['prod_nz'   ].transform(lambda s: s.shift(1).expanding().mean())
feat = feat.merge(prod[['NEST_NAME', 'PRODUCT_NAME', 'week_start',
                        'product_hist_mean', 'product_hist_nz_rate']],
                  on=['NEST_NAME', 'PRODUCT_NAME', 'week_start'], how='left')

# Pair-level expanding history
feat = feat.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
g_pair = feat.groupby(PAIR_COLS, sort=False)['weekly_units']
feat['fp_hist_mean'] = g_pair.transform(lambda s: s.shift(1).expanding().mean())
feat['fp_nz_tmp']    = (feat['weekly_units'] > 0).astype(int)
feat['fp_hist_nz_rate'] = feat.groupby(PAIR_COLS, sort=False)['fp_nz_tmp'].transform(lambda s: s.shift(1).expanding().mean())
feat = feat.drop(columns=['fp_nz_tmp'])
print(f'Expanding history done. elapsed={time.time()-t0:.1f}s')


Expanding history done. elapsed=385.9s


### 7f. Label encoding (only `HEALTH_FACILITY_NAME` and `PRODUCT_NAME`)

In [19]:
for col in ['HEALTH_FACILITY_NAME', 'PRODUCT_NAME']:
    feat[col + '_enc'] = LabelEncoder().fit_transform(feat[col].astype(str))
print(f'Encodings done. elapsed={time.time()-t0:.1f}s')
print('Unified feature frame shape:', feat.shape)
print('Per-nest rows in feat:')
print(feat['NEST_NAME'].value_counts(dropna=False))


Encodings done. elapsed=390.2s
Unified feature frame shape: (687678, 48)
Per-nest rows in feat:
NEST_NAME
RW1 Muhanga    245226
GH3 Vobsi      211566
RW2 Kayonza    136758
GH1 Omenako     94128
Name: count, dtype: int64


## Step 8 — Save the unified feature table (parquet)

In [20]:
OUT_DIR = Path('/content')
FEATURE_COLS = [
    # calendar
    'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
    # lags
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
    # rollings
    'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
    'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
    'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
    # intermittent
    'weeks_since_last_demand', 'last_nonzero_units',
    'historical_nonzero_rate', 'cumulative_active_weeks',
    # expanding history
    'facility_hist_mean', 'facility_hist_nz_rate',
    'product_hist_mean',  'product_hist_nz_rate',
    'fp_hist_mean',       'fp_hist_nz_rate',
    # pair-level numerics
    'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
    # encodings
    'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
]
KEEP_COLS = (PAIR_COLS + ['week_start', 'weekly_units', 'demand_class', 'tier', 'split']
             + FEATURE_COLS)
unified = feat[KEEP_COLS].copy()
unified.to_parquet(OUT_DIR / 'unified_features.parquet', index=False)
print(f'Saved unified_features.parquet  shape: {unified.shape}'
      f'  size={(OUT_DIR / "unified_features.parquet").stat().st_size/1e6:.1f} MB')
print(f'\nTOTAL elapsed: {time.time()-t0:.1f}s')
print(f'\nFeature columns ({len(FEATURE_COLS)}):')
for c in FEATURE_COLS:
    print(f'  - {c}')

Saved unified_features.parquet  shape: (687678, 48)  size=11.5 MB

TOTAL elapsed: 398.1s

Feature columns (40):
  - week_of_year
  - month
  - quarter
  - week_sin
  - week_cos
  - rainy_season
  - lag_1
  - lag_2
  - lag_3
  - lag_4
  - lag_8
  - lag_12
  - roll_mean_4
  - roll_std_4
  - roll_max_4
  - roll_nz_rate_4
  - roll_mean_8
  - roll_std_8
  - roll_max_8
  - roll_nz_rate_8
  - roll_mean_13
  - roll_std_13
  - roll_max_13
  - roll_nz_rate_13
  - weeks_since_last_demand
  - last_nonzero_units
  - historical_nonzero_rate
  - cumulative_active_weeks
  - facility_hist_mean
  - facility_hist_nz_rate
  - product_hist_mean
  - product_hist_nz_rate
  - fp_hist_mean
  - fp_hist_nz_rate
  - adi
  - cv2
  - mean_nonzero
  - nonzero_weeks
  - HEALTH_FACILITY_NAME_enc
  - PRODUCT_NAME_enc
